In [ ]:
import pandas as pd
from pathlib import Path
import os
import xarray as xr

In [4]:
PNNL_DIR = Path("/data0/skagit_met/data_transfer/data/PNNL/historical")

In [ ]:
window_dates = set()
for dt in event_dates:
    for i in range(6, -1, -1):
        window_dates.add(dt - pd.Timedelta(days=i))
window_dates = sorted(list(window_dates))

In [ ]:
year = 2006

In [ ]:
pnnl_path = os.path.join(PNNL_DIR, str(year), f"PNNL_WRF.HIST.CTRL.hourly.PREC_ACC_NC.{year}.nc")
if os.path.exists(pnnl_path):
    try:
        ds = xr.open_dataset(pnnl_path)
        hourly_dates = []
        for d in window_dates:
            hourly_dates.extend(pd.date_range(d, d + pd.Timedelta(hours=23), freq='h'))
        available_hourly = [d for d in hourly_dates if d in ds.time.values]
        if available_hourly:
            da_subset = ds['PREC_ACC_NC'].sel(time=available_hourly)
            s_hourly = calculate_basin_mean(da_subset, masks_2d['PNNL'])
            pnnl_daily = s_hourly.resample('1D').sum()
            pnnl_daily.index = pnnl_daily.index.normalize()